# Instalacion de librerias (se trabaja con Python 3.10 compatible con tensorflow)

In [ ]:
pip install tensorflow opencv-python numpy matplotlib scikit-learn pydicom Pillow 

# Importacion de las librerias globales
- tensorflow / keras: Es la biblioteca principal para construir y entrenar redes neuronales (Deep Learning). Keras es una API de alto nivel dentro de TensorFlow que simplifica mucho el proceso.
- ImageDataGenerator: Una herramienta de Keras para cargar imágenes desde directorios, preprocesarlas (como normalizar píxeles) y aplicar aumento de datos (data augmentation) en tiempo real.
- MobileNetV2 (u otros como VGG16, ResNet50): Son arquitecturas de Redes Neuronales Convolucionales (CNN) pre-entrenadas en un conjunto de datos masivo (generalmente ImageNet). Usarlas se llama Transfer Learning. Nos ahorra tiempo y a menudo mejora el rendimiento, especialmente con datasets pequeños.
- Dense, GlobalAveragePooling2D, Dropout, Input: Tipos de capas que usaremos para construir nuestra red neuronal sobre el modelo base pre-entrenado.
        - Input: Define la forma de los datos de entrada.
        GlobalAveragePooling2D: Reduce la dimensionalidad espacial de los mapas de características de la CNN, preparando la salida para la capa de clasificación.
        - Dense: Capa neuronal "normal", completamente conectada. La usaremos para la clasificación final.
        Dropout: Técnica de regularización para prevenir el sobreajuste (overfitting), "apagando" aleatoriamente neuronas durante el entrenamiento.
    - Model: Clase para definir el modelo Keras completo, especificando las capas de entrada y salida.
    - Adam: Un algoritmo de optimización popular y efectivo que ajusta los pesos del modelo durante el entrenamiento para minimizar el error (loss).
    - ModelCheckpoint, EarlyStopping: Son "Callbacks", funciones que se ejecutan en ciertos puntos durante el entrenamiento.
        - ModelCheckpoint: Guarda el modelo (generalmente, el mejor modelo encontrado hasta el momento según una métrica).
        - EarlyStopping: Detiene el entrenamiento si el rendimiento en el conjunto de validación deja de mejorar, evitando el sobreajuste y ahorrando tiempo.
    - matplotlib.pyplot: Biblioteca para generar gráficos (usada aquí para visualizar el historial de entrenamiento).
    - numpy: Biblioteca fundamental para operaciones numéricas en Python, especialmente con arrays (que es como se representan las imágenes y los datos en TensorFlow)
    - os: Módulo para interactuar con el sistema operativo, usado aquí para verificar si existen las rutas a los datos.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2 # O cualquier otro modelo pre-entrenado
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuración
- IMG_WIDTH, IMG_HEIGHT: Define el tamaño (ancho y alto en píxeles) al que se redimensionarán todas las imágenes antes de pasarlas al modelo. Este tamaño suele depender del modelo pre-entrenado que uses (MobileNetV2 espera 224x224 por defecto).
- BATCH_SIZE: Número de imágenes que se procesan juntas en cada paso del entrenamiento. Un batch size más grande puede acelerar el entrenamiento (mejor uso de GPU) pero consume más memoria. Un tamaño común es 32 o 64.
- EPOCHS: Número de veces que el modelo verá el conjunto de datos de entrenamiento completo. Demasiadas épocas pueden llevar a sobreajuste; muy pocas, a subajuste (underfitting). EarlyStopping ayuda a encontrar un buen punto.
- NUM_CLASSES: El número total de categorías que quieres clasificar (en tu caso: Normal, Litiasis, ERC, Absceso = 4).
- LEARNING_RATE: Controla cuán grandes son los ajustes a los pesos del modelo en cada paso de optimización. Un valor pequeño es común en Transfer Learning o Fine-tuning.
- TRAIN_DIR, VALIDATION_DIR: Rutas a las carpetas que contienen las imágenes de entrenamiento y validación, organizadas por subcarpetas de clase.
- MODEL_SAVE_PATH: Nombre y ruta del archivo donde se guardará el modelo entrenado (el mejor, gracias a ModelCheckpoint). El formato .h5 es un formato común de Keras para guardar la arquitectura, los pesos y el estado del optimizador.

In [ ]:
IMG_WIDTH, IMG_HEIGHT = 224, 224  # Tamaño esperado por MobileNetV2
BATCH_SIZE = 32
EPOCHS = 50 # Ajustar según sea necesario
NUM_CLASSES = 4 # Normal, LitiasisRenal, EnfermedadRenalCronica, AbscesoRenal
LEARNING_RATE = 0.0001

# Rutas a tus datos (¡Asegúrate que existan!)
TRAIN_DIR = '/path/to/your/dataset/train'
VALIDATION_DIR = '/path/to/your/dataset/validation'
MODEL_SAVE_PATH = './renal_disease_classifier_mobilenetv2.h5' # Donde se guardará el modelo

# Verificar si los directorios existen
if not os.path.exists(TRAIN_DIR):
    raise ValueError(f"Directorio de entrenamiento no encontrado: {TRAIN_DIR}")
if not os.path.exists(VALIDATION_DIR):
    raise ValueError(f"Directorio de validación no encontrado: {VALIDATION_DIR}")



# Se crean dos generadores: uno para entrenamiento y otro para validación.
- rescale=1./255: Normaliza los valores de los píxeles de las imágenes (que originalmente van de 0 a 255) a un rango entre 0 y 1. Esto es crucial para que la red neuronal funcione correctamente.
- Parámetros de Aumento (solo en train_datagen): rotation_range, width_shift_range, height_shift_range, shear_range, zoom_range, horizontal_flip. Estas opciones aplican transformaciones aleatorias (rotaciones, desplazamientos, zoom, etc.) a las imágenes de entrenamiento cada vez que se cargan.
- Propósito del Aumento: Crea variaciones artificiales de las imágenes existentes, haciendo que el dataset parezca más grande y diverso. Ayuda enormemente a que el modelo generalice mejor (funcione bien con imágenes nuevas que no ha visto) y reduce el sobreajuste.
- validation_datagen NO tiene aumento: Es fundamental evaluar el modelo en datos "limpios", tal como se esperarían en el mundo real, sin las variaciones artificiales del aumento. Solo se aplica la normalización (rescale).

In [ ]:
# --- 2. Preprocesamiento y Aumento de Datos ---
# Normalización y aumento para el conjunto de entrenamiento
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalizar pixeles a [0, 1]
    rotation_range=20,           # Aumento: rotaciones aleatorias
    width_shift_range=0.2,       # Aumento: desplazamientos horizontales
    height_shift_range=0.2,      # Aumento: desplazamientos verticales
    shear_range=0.2,             # Aumento: transformaciones de corte
    zoom_range=0.2,              # Aumento: zoom aleatorio
    horizontal_flip=True,        # Aumento: volteo horizontal
    fill_mode='nearest'          # Estrategia para rellenar pixeles nuevos
)

# Solo normalización para el conjunto de validación (¡NO AUMENTAR!)
validation_datagen = ImageDataGenerator(rescale=1./255)

# Generadores de datos
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical' # Para clasificación multiclase
)

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # No es necesario barajar la validación
)

# Verificar clases encontradas
print("Clases encontradas (Entrenamiento):", train_generator.class_indices)
print("Clases encontradas (Validación):", validation_generator.class_indices)
if len(train_generator.class_indices) != NUM_CLASSES:
     print(f"ADVERTENCIA: Se esperaban {NUM_CLASSES} clases, pero se encontraron {len(train_generator.class_indices)} en {TRAIN_DIR}")
     # Considera detener el script o ajustar NUM_CLASSES si esto es un error grave



# Estos métodos usan los ImageDataGenerator para leer imágenes directamente desde las carpetas (TRAIN_DIR, VALIDATION_DIR).
Automáticamente infieren las clases a partir de los nombres de las subcarpetas.
- target_size: Asegura que todas las imágenes se redimensionen al tamaño especificado (IMG_WIDTH, IMG_HEIGHT).
- batch_size: Indica cuántas imágenes cargar en cada lote.
- class_mode='categorical': Indica que es un problema de clasificación multiclase y que las etiquetas deben ser convertidas a formato one-hot encoding (ej: [0, 1, 0, 0] para la segunda clase de 4). Esto es necesario para usar la función de pérdida categorical_crossentropy.
- shuffle=False (en validation_generator): No baraja las imágenes de validación para tener una evaluación consistente época tras época.

- Transfer Learning: Se carga MobileNetV2 con pesos pre-entrenados en ImageNet (weights='imagenet') y sin su capa final de clasificación (include_top=False).
- Congelación (base_model.trainable = False): Se "congelan" los pesos del modelo base. Esto significa que durante la fase inicial del entrenamiento, solo se ajustarán los pesos de las capas nuevas que añadimos nosotros. Esto evita que la información valiosa aprendida por MobileNetV2 se destruya al principio con gradientes potencialmente grandes de las capas nuevas no entrenadas.
Capas Nuevas
    - GlobalAveragePooling2D: Reduce la salida del base_model a un vector de características por imagen.
    - Dropout: Ayuda a prevenir el sobreajuste en nuestras nuevas capas.
    - Dense(num_classes, activation='softmax'): La capa de salida final. Tiene tantas neuronas como clases (NUM_CLASSES) y usa la activación softmax para producir probabilidades para cada clase (la suma de probabilidades será 1).
    - Model(inputs, outputs): Define el modelo final conectando la entrada (Input) con la salida (Dense) a través de las capas intermedias (incluyendo el base_model congelado).

In [ ]:
# --- 3. Construcción del Modelo (Transfer Learning) ---
def build_model(num_classes):
    # Cargar MobileNetV2 pre-entrenado en ImageNet, sin la capa superior de clasificación
    base_model = MobileNetV2(weights='imagenet', include_top=False,
                             input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))

    # Congelar las capas del modelo base para no re-entrenarlas inicialmente
    base_model.trainable = False

    # Añadir nuestras propias capas de clasificación
    inputs = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    # Asegurarse que el preprocesamiento del modelo base se aplique si es necesario
    # (MobileNetV2 usa rescale 1./127.5 - 1, pero ya hicimos 1./255, podríamos ajustar o usar tf.keras.applications.mobilenet_v2.preprocess_input)
    # Para simplificar aquí, asumimos que 1./255 es suficiente o que ajustaremos el preprocesamiento del generador si es necesario.
    x = base_model(inputs, training=False) # Importante: training=False para capas congeladas (BatchNorm)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x) # Dropout para regularización
    outputs = Dense(num_classes, activation='softmax')(x) # Softmax para multiclase

    model = Model(inputs, outputs)
    return model

model = build_model(NUM_CLASSES)
model.summary() # Imprime la arquitectura del modelo



# Prepara el modelo para el entrenamiento.
- optimizer: Especifica cómo se actualizarán los pesos (Adam con la tasa de aprendizaje definida).
- loss: Define la función que mide qué tan erróneas son las predicciones del modelo. categorical_crossentropy es la estándar para clasificación
- multiclase con etiquetas one-hot. El objetivo del entrenamiento es minimizar esta función de pérdida.
- metrics: Lista de métricas para evaluar el rendimiento del modelo durante el entrenamiento y la validación. accuracy (precisión: porcentaje de predicciones correctas) es la más común para empezar.

In [ ]:
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy', # Loss para clasificación multiclase
              metrics=['accuracy'])

Se configuran los callbacks:
- ModelCheckpoint: Para guardar automáticamente el modelo en MODEL_SAVE_PATH solo si la val_accuracy (precisión en el conjunto de validación) mejora (mode='max').
- EarlyStopping: Para monitorizar val_loss (pérdida en validación). Si no mejora (mode='min' implícito para loss) durante un número determinado de épocas (patience=10), el entrenamiento se detiene. restore_best_weights=True asegura que el modelo final tenga los pesos del mejor epoch (según val_loss), no necesariamente del último.
Se agrupan en una lista para pasarlos a model.fit.

In [ ]:
# Guardar el mejor modelo basado en la precisión de validación
checkpoint = ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', verbose=1,
                             save_best_only=True, mode='max')

# Detener el entrenamiento si la pérdida de validación no mejora
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, # Aumentar 'patience' si es necesario
                               restore_best_weights=True) # Restaura los pesos del mejor epoch

callbacks_list = [checkpoint, early_stopping]


# Esta es la línea que inicia el proceso de entrenamiento
- Le dice al modelo que entrene usando los datos del train_generator.
- epochs: Durante cuántas épocas entrenar (a menos que EarlyStopping lo detenga antes).
- validation_data: Usa validation_generator para evaluar el modelo al final de cada época. Esto es crucial para monitorizar el sobreajuste y para que los callbacks funcionen correctamente.
- steps_per_epoch: Cuántos batches del train_generator constituyen una época completa. Generalmente numero_total_imagenes_entrenamiento / BATCH_SIZE.
- validation_steps: Similar, pero para el conjunto de validación.
- callbacks: Usa la lista de callbacks definida anteriormente.
- La función fit devuelve un objeto history que contiene un registro de la pérdida (loss) y las métricas (accuracy) para cada época, tanto en el conjunto de entrenamiento como en el de validación.

In [ ]:
print("Iniciando entrenamiento...")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples
    callbacks=callbacks_list
)
print("Entrenamiento completado.")

In [ ]:
# Aunque ModelCheckpoint guarda el mejor, podemos guardar el estado final
# o el estado después del fine-tuning si se realizó.
# Si usaste EarlyStopping con restore_best_weights=True, este modelo
# será el mismo que el guardado por ModelCheckpoint (el mejor según val_loss).
# Si no, este será el modelo del último epoch.

print(f"Guardando el modelo final en: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print("Modelo final guardado.")

# También es útil guardar el mapeo de índices a etiquetas de clase
import json
class_indices = train_generator.class_indices # O validation_generator.class_indices
labels = dict((v,k) for k,v in class_indices.items())

labels_path = os.path.splitext(MODEL_SAVE_PATH)[0] + '_labels.json'
with open(labels_path, 'w') as f:
    json.dump(labels, f, indent=4)
print(f"Mapeo de etiquetas guardado en: {labels_path}")



Esta sección describe un paso avanzado pero no esta funcionando adecuadamente.

- Concepto: Después de que el modelo ha entrenado las capas superiores congelando el base_model, se puede "descongelar" una parte del base_model (base_model.trainable = True, luego se congelan las capas inferiores) y continuar el entrenamiento con una tasa de aprendizaje muy baja.
- Propósito: Permite ajustar ligeramente los pesos pre-entrenados del base_model para que se adapten mejor a las características específicas de tu dataset de imágenes renales, potencialmente mejorando aún más el rendimiento.
- Precaución: Requiere una tasa de aprendizaje muy baja para no destruir las características pre-entrenadas.

In [ ]:
# --- 7. (Opcional) Descongelar algunas capas para Fine-tuning ---
# Esto se hace DESPUÉS de que el modelo inicial haya convergido un poco
# print("Iniciando Fine-tuning...")
# base_model.trainable = True
#
# # Congelar la mayoría de las capas, descongelar solo las últimas
# fine_tune_at = 100 # Ejemplo: descongelar desde la capa 100 en adelante
# for layer in base_model.layers[:fine_tune_at]:
#     layer.trainable = False
#
# # Re-compilar con una tasa de aprendizaje MUY BAJA
# model.compile(optimizer=Adam(learning_rate=LEARNING_RATE / 10),
#               loss='categorical_crossentropy',
#               metrics=['accuracy'])
#
# # Continuar entrenamiento (fine-tuning)
# history_fine = model.fit(
#     train_generator,
#     steps_per_epoch=train_generator.samples // BATCH_SIZE,
#     epochs=EPOCHS + 10, # Entrenar por unas pocas épocas más
#     initial_epoch=history.epoch[-1] + 1, # Continuar desde donde se quedó
#     validation_data=validation_generator,
#     validation_steps=validation_generator.samples // BATCH_SIZE,
#     callbacks=callbacks_list # Puedes reutilizar o ajustar callbacks
# )
# print("Fine-tuning completado.")




# Visualización del Entrenamiento. Esta función toma el objeto history devuelto por model.fit

Extrae los valores de precisión y pérdida de entrenamiento y validación para cada época.  Usa matplotlib para crear dos gráficos:
- Grafico de Precisión (Training vs Validation) a lo largo de las épocas.
- Grafico de Pérdida (Training vs Validation) a lo largo de las épocas.
Propósito: Permite analizar visualmente cómo progresó el entrenamiento. Se puede detectar:
- Sobreajuste: Si la precisión de entrenamiento sigue subiendo mientras la de validación se estanca o baja (o la pérdida de entrenamiento baja y la de validación sube).
- Subajuste: Si ambas precisiones (entrenamiento y validación) son bajas.
- Convergencia: Si las curvas se estabilizan.

In [ ]:
# --- 8. (Opcional) Visualización del Entrenamiento ---
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc)) # Usar la longitud real de la historia

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')


    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')

    plt.tight_layout()
    plt.show()

# Graficar historial principal
plot_history(history)
